In [1]:
import cv2
import mediapipe as mp
import numpy as np
import torch

In [2]:
def standardize_data(data):

    mean = data.mean(axis = (1, 2), keepdims = True)

    std = data.std(axis = (1, 2), keepdims = True)

    standardized_data = (data - mean) / (std + 1e-8)

    return standardized_data

In [3]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(2)

model_path = "./lstm_model_scripted.pt"
model = torch.jit.load(model_path)
model.eval()

labels_map = {0: "Normal", 1: "Warning", 2: "Fall"}

is_collecting = True
keypoints_list = []

while cap.isOpened():

    keypoints = []

    ret, frame = cap.read()

    if not ret:
        print("Cannot open video or video is terminated")
        break

    frame = cv2.resize(frame, (1024, 768))

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    result = pose.process(image)

    if result.pose_landmarks:

        mp_drawing.draw_landmarks(frame, result.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color = (255, 0, 0), thickness = 2),
                                  mp_drawing.DrawingSpec(color = (255, 255, 255), thickness = 2))
        
        kp = [[lmk.x, lmk.y, lmk.z] for lmk in result.pose_landmarks.landmark]

        for key in kp:
            for k in key:
                keypoints.append(k)

        if is_collecting:
            keypoints_list.append(keypoints)

        status_text = "Collecting" if is_collecting else "Paused"

        is_collecting = not is_collecting

        if not is_collecting and keypoints_list:

            keypoints_array = np.array(keypoints_list)

            keypoints_standardized = standardize_data(np.expand_dims(keypoints_array, axis = 0))[0]

            input_data = torch.tensor(keypoints_standardized).float()
            input_data = input_data.view(1, keypoints_standardized.shape[0], -1)

            with torch.no_grad():
                outputs = model(input_data)
                probabilities = torch.softmax(outputs, dim = 1)

                predicted_class = torch.argmax(probabilities, dim=1).item()

                confidence = probabilities[0, predicted_class].item()

        predicted_label = labels_map[predicted_class]
        confidence_percent = confidence * 100
        cv2.putText(frame, f"Predicted: {predicted_label}, Confidence: {confidence_percent:.2f}%", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 64, 0), 2)

        cv2.putText(frame, "Probabilities: ", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 64, 64), 2)

        cv2.putText(frame, f"{labels_map[0]}: " + f"{round(float(probabilities[0][0]), 2)}", (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 64, 64), 2)
        cv2.putText(frame, f"{labels_map[1]}: " + f"{round(float(probabilities[0][1]), 2)}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 64, 64), 2)
        cv2.putText(frame, f"{labels_map[2]}: " + f"{round(float(probabilities[0][2]), 2)}", (10, 190), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 64, 64), 2)


        keypoints_list = []

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1757920211.699048  756538 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1757920211.701506  756597 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1757920211.755958  756582 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757920211.780486  756590 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757920212.177319  756582 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


KeyboardInterrupt: 